# Preprocessing Pipeline (Production-Ready)

This notebook creates reproducible ILPD preprocessing artifacts from raw data and saves outputs under the repository `data/` directory.

## Outputs
- `data/interim/ILPD_cleaned.csv`
- `data/processed/ILPD_clinically_capped.csv`
- `data/processed/ILPD_robust_scaled_features.csv`
- `produced_reports/docs/ILPD_preprocessing_metadata.json`

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler


In [2]:
def resolve_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() or (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root from current working directory")

ROOT = resolve_repo_root(Path.cwd())
RAW_PATH = ROOT / "data" / "raw" / "ILPD.csv"
INTERIM_DIR = ROOT / "data" / "interim"
PROCESSED_DIR = ROOT / "data" / "processed"
REPORTS_DIR = ROOT / "produced_reports" / "docs"

for directory in [INTERIM_DIR, PROCESSED_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Root: {ROOT}")
print(f"Raw dataset: {RAW_PATH}")
print(f"Interim output dir: {INTERIM_DIR}")
print(f"Processed output dir: {PROCESSED_DIR}")
print(f"Reports output dir: {REPORTS_DIR}")


Root: /Users/samyabrataroy/Downloads/Spring_Internship_2026
Raw dataset: /Users/samyabrataroy/Downloads/Spring_Internship_2026/data/raw/ILPD.csv
Interim output dir: /Users/samyabrataroy/Downloads/Spring_Internship_2026/data/interim
Processed output dir: /Users/samyabrataroy/Downloads/Spring_Internship_2026/data/processed
Reports output dir: /Users/samyabrataroy/Downloads/Spring_Internship_2026/produced_reports/docs


## 1) Load and standardize schema

In [3]:
COLUMN_NAMES = [
    "Age",
    "Gender",
    "Total_Bilirubin",
    "Direct_Bilirubin",
    "Alkaline_Phosphotase",
    "Alamine_Aminotransferase",
    "Aspartate_Aminotransferase",
    "Total_Proteins",
    "Albumin",
    "Albumin_and_Globulin_Ratio",
    "Result",
]

df = pd.read_csv(RAW_PATH, header=None, names=COLUMN_NAMES)
print(df.shape)
df.head()


(583, 11)


,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Proteins,Albumin,Albumin_and_Globulin_Ratio,Result
0,65,Female,0.7,0.1,187,16,18,6.8,3.3,0.90,1
1,62,Male,10.9,5.5,699,64,100,7.5,3.2,0.74,1
2,62,Male,7.3,4.1,490,60,68,7.0,3.3,0.89,1
3,58,Male,1.0,0.4,182,14,20,6.8,3.4,1.00,1
4,72,Male,3.9,2.0,195,27,59,7.3,2.4,0.40,1


## 2) Missing/invalid/infinite checks + duplicate removal + gender encoding

This step performs data quality checks first, encodes `Gender`, and removes duplicate rows before missing-value treatment.

In [4]:
missing_before = df.isnull().sum().to_dict()

# Invalid/infinite checks BEFORE missing-value treatment
invalid_checks = {
    "Age_negative": int((df["Age"] < 0).sum()),
    "Age_too_high": int((df["Age"] > 120).sum()),
    "Bilirubin_negative": int((df["Total_Bilirubin"] < 0).sum()),
    "Proteins_negative": int((df["Total_Proteins"] < 0).sum()),
    "Albumin_negative": int((df["Albumin"] < 0).sum()),
    "AG_ratio_negative": int((df["Albumin_and_Globulin_Ratio"] < 0).sum()),
    "Infinite_numeric_values": int(np.isinf(df.select_dtypes(include=[np.number])).sum().sum()),
}

# Encode categorical column for modeling compatibility
df["Gender"] = df["Gender"].map({"Male": 0, "Female": 1})

# Remove exact duplicates
duplicates_removed = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

print("Missing (before):", missing_before)
print("Invalid checks:", invalid_checks)
print("Duplicates removed:", duplicates_removed)
print("Shape after dedup:", df.shape)

Missing (before): {'Age': 0, 'Gender': 0, 'Total_Bilirubin': 0, 'Direct_Bilirubin': 0, 'Alkaline_Phosphotase': 0, 'Alamine_Aminotransferase': 0, 'Aspartate_Aminotransferase': 0, 'Total_Proteins': 0, 'Albumin': 0, 'Albumin_and_Globulin_Ratio': 4, 'Result': 0}
Invalid checks: {'Age_negative': 0, 'Age_too_high': 0, 'Bilirubin_negative': 0, 'Proteins_negative': 0, 'Albumin_negative': 0, 'AG_ratio_negative': 0, 'Infinite_numeric_values': 0}
Duplicates removed: 13
Shape after dedup: (570, 11)


## 3) Missing value treatment with group-wise median

Single selected strategy for `Albumin_and_Globulin_Ratio`:
- group-wise median by `Gender`
- global median fallback

In [5]:
impute_col = "Albumin_and_Globulin_Ratio"
imputation_strategy = "group_median_by_gender_with_global_fallback"

global_median = df[impute_col].median()
df[impute_col] = df.groupby("Gender")[impute_col].transform(lambda s: s.fillna(s.median()))
df[impute_col] = df[impute_col].fillna(global_median)

missing_after = df.isnull().sum().to_dict()

print("Imputation strategy:", imputation_strategy)
print("Missing (after):", missing_after)
print("Shape after imputation:", df.shape)

Imputation strategy: group_median_by_gender_with_global_fallback
Missing (after): {'Age': 0, 'Gender': 0, 'Total_Bilirubin': 0, 'Direct_Bilirubin': 0, 'Alkaline_Phosphotase': 0, 'Alamine_Aminotransferase': 0, 'Aspartate_Aminotransferase': 0, 'Total_Proteins': 0, 'Albumin': 0, 'Albumin_and_Globulin_Ratio': 0, 'Result': 0}
Shape after imputation: (570, 11)


## 4) Clinically bounded capping (winsorization-style)

Caps only biologically implausible extremes while preserving records.

In [6]:
clinical_limits = {
    "Age": (0, 100),
    "Total_Bilirubin": (0, 50),
    "Direct_Bilirubin": (0, 30),
    "Alkaline_Phosphotase": (0, 2000),
    "Alamine_Aminotransferase": (0, 5000),
    "Aspartate_Aminotransferase": (0, 10000),
    "Total_Proteins": (0, 15),
    "Albumin": (0, 6.0),
    "Albumin_and_Globulin_Ratio": (0, 3.0),
}

capped_df = df.copy()
for col, (min_val, max_val) in clinical_limits.items():
    if col in capped_df.columns:
        capped_df[col] = capped_df[col].clip(lower=min_val, upper=max_val)

capped_df.head()


,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Proteins,Albumin,Albumin_and_Globulin_Ratio,Result
0,65,1,0.7,0.1,187,16,18,6.8,3.3,0.90,1
1,62,0,10.9,5.5,699,64,100,7.5,3.2,0.74,1
2,62,0,7.3,4.1,490,60,68,7.0,3.3,0.89,1
3,58,0,1.0,0.4,182,14,20,6.8,3.4,1.00,1
4,72,0,3.9,2.0,195,27,59,7.3,2.4,0.40,1


## 5) Robust-scaled feature dataset (for modeling convenience)

`Result` is retained separately and not scaled.

In [7]:
feature_cols = [c for c in capped_df.columns if c != "Result"]

scaler = RobustScaler()
scaled_features = pd.DataFrame(
    scaler.fit_transform(capped_df[feature_cols]),
    columns=feature_cols,
)

scaled_with_target = scaled_features.copy()
scaled_with_target["Result"] = capped_df["Result"].values

scaled_with_target.head()


,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Proteins,Albumin,Albumin_and_Globulin_Ratio,Result
0,0.80,1.0,-0.166667,-0.181818,-0.172131,-0.513514,-0.372470,0.142857,0.166667,-0.125,1
1,0.68,0.0,5.500000,4.727273,4.024590,0.783784,0.955466,0.642857,0.083333,-0.525,1
2,0.68,0.0,3.500000,3.454545,2.311475,0.675676,0.437247,0.285714,0.166667,-0.150,1
3,0.52,0.0,0.000000,0.090909,-0.213115,-0.567568,-0.340081,0.142857,0.250000,0.125,1
4,1.08,0.0,1.611111,1.545455,-0.106557,-0.216216,0.291498,0.500000,-0.583333,-1.375,1


## 6) Persist pipeline artifacts to `data/`

In [8]:
clean_path = INTERIM_DIR / "ILPD_cleaned.csv"
capped_path = PROCESSED_DIR / "ILPD_clinically_capped.csv"
scaled_path = PROCESSED_DIR / "ILPD_robust_scaled_features.csv"
metadata_path = REPORTS_DIR / "ILPD_preprocessing_metadata.json"

df.to_csv(clean_path, index=False)
capped_df.to_csv(capped_path, index=False)
scaled_with_target.to_csv(scaled_path, index=False)

metadata = {
    "input": str(RAW_PATH),
    "rows_raw": int(pd.read_csv(RAW_PATH, header=None).shape[0]),
    "rows_cleaned": int(df.shape[0]),
    "columns": list(df.columns),
    "missing_before": missing_before,
    "missing_after": missing_after,
    "imputation_strategy": imputation_strategy,
    "invalid_checks": invalid_checks,
    "duplicates_removed": duplicates_removed,
    "outputs": {
        "cleaned": str(clean_path),
        "clinically_capped": str(capped_path),
        "robust_scaled_features": str(scaled_path),
    },
}

metadata_path.write_text(json.dumps(metadata, indent=2))

print("Saved:")
print(clean_path)
print(capped_path)
print(scaled_path)
print(metadata_path)


Saved:
/Users/samyabrataroy/Downloads/Spring_Internship_2026/data/interim/ILPD_cleaned.csv
/Users/samyabrataroy/Downloads/Spring_Internship_2026/data/processed/ILPD_clinically_capped.csv
/Users/samyabrataroy/Downloads/Spring_Internship_2026/data/processed/ILPD_robust_scaled_features.csv
/Users/samyabrataroy/Downloads/Spring_Internship_2026/produced_reports/docs/ILPD_preprocessing_metadata.json


## Export notebook to HTML report

In [9]:
import subprocess

HTML_DIR = ROOT / "produced_reports"
HTML_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK_PATH = ROOT / "scripts" / "03_preprocessing_pipeline.ipynb"

subprocess.run([
    "jupyter", "nbconvert",
    "--to", "html",
    str(NOTEBOOK_PATH),
    "--output-dir", str(HTML_DIR),
], check=True)

print(f"Exported HTML to: {HTML_DIR / NOTEBOOK_PATH.with_suffix('.html').name}")

[NbConvertApp] Converting notebook /Users/samyabrataroy/Downloads/Spring_Internship_2026/scripts/03_preprocessing_pipeline.ipynb to html


Exported HTML to: /Users/samyabrataroy/Downloads/Spring_Internship_2026/produced_reports/03_preprocessing_pipeline.html


[NbConvertApp] Writing 306353 bytes to /Users/samyabrataroy/Downloads/Spring_Internship_2026/produced_reports/03_preprocessing_pipeline.html
